# 09 — Hybrid model: XGBoost (static) + LSTM (sequence)

The project's headline model. Two branches, each doing what it is best at, fused:

```
static features ─> XGBoost ─────────────> risk score (1 number) ┐
                                                                 concat ─> MLP ─> PD
payment sequence ─> LSTM ─> temporal embedding (32 numbers) ─────┘
```

Data: Taiwan Credit Card. Static branch = credit limit, age, bill/payment amounts. Temporal branch =
the 6-month payment-status sequence.

> **Run the XGBoost branch first** (separate process, torch+xgboost segfault together):
> `uv run python src/make_hybrid_features.py`  — it writes `data/processed/hybrid_feats.npz`.
> XGBoost train-row scores are **out-of-fold** (5-fold CV) so the fusion never sees leaked scores.

## 1. Load the precomputed features

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score
torch.manual_seed(42); np.random.seed(42)

d = np.load("../data/processed/hybrid_feats.npz")
seq_tr,seq_va,seq_te = [torch.tensor(d[k]) for k in ("seq_tr","seq_va","seq_te")]
s_tr,s_va,s_te       = [torch.tensor(d[k]).unsqueeze(1) for k in ("s_tr","s_va","s_te")]  # xgb score (N,1)
y_tr,y_va,y_te       = [d[k] for k in ("y_tr","y_va","y_te")]
ytr_t = torch.tensor(y_tr).unsqueeze(1)
print("sequences:", seq_tr.shape, "| xgb score per borrower:", s_tr.shape)
print("XGBoost static-only test AUC (from the script):", round(float(d["xgb_auc"]),4))

sequences: torch.Size([18000, 6, 1]) | xgb score per borrower: torch.Size([18000, 1])
XGBoost static-only test AUC (from the script): 0.7298


## 2. Two models: LSTM alone, and the hybrid

The **hybrid** runs the sequence through an LSTM to a 32-dim embedding, concatenates the single
XGBoost score, and passes the 33 numbers through a small MLP.

In [2]:
spw=torch.tensor([(y_tr==0).sum()/(y_tr==1).sum()])
crit=nn.BCEWithLogitsLoss(pos_weight=spw)

def train(model, uses_score):
    opt=torch.optim.Adam(model.parameters(), lr=5e-3)
    ld=DataLoader(TensorDataset(seq_tr,s_tr,ytr_t), batch_size=256, shuffle=True)
    best,bs=0,None
    for ep in range(30):
        model.train()
        for xb,sb,yb in ld:
            opt.zero_grad()
            out=model(xb,sb) if uses_score else model(xb)
            crit(out,yb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            pv=torch.sigmoid(model(seq_va,s_va) if uses_score else model(seq_va)).numpy().ravel()
        va=roc_auc_score(y_va,pv)
        if va>best: best,bs=va,{k:v.clone() for k,v in model.state_dict().items()}
    model.load_state_dict(bs); model.eval()
    with torch.no_grad():
        pt=torch.sigmoid(model(seq_te,s_te) if uses_score else model(seq_te)).numpy().ravel()
    return roc_auc_score(y_te,pt)

class LSTMAlone(nn.Module):
    def __init__(s,h=32):
        super().__init__(); s.lstm=nn.LSTM(1,h,batch_first=True); s.head=nn.Linear(h,1)
    def forward(s,x):
        o,(hn,cn)=s.lstm(x); return s.head(hn[-1])

class Hybrid(nn.Module):
    def __init__(s,h=32):
        super().__init__()
        s.lstm=nn.LSTM(1,h,batch_first=True)
        s.head=nn.Sequential(nn.Linear(h+1,16), nn.ReLU(), nn.Linear(16,1))   # +1 = xgb score
    def forward(s,x,score):
        o,(hn,cn)=s.lstm(x); emb=hn[-1]
        return s.head(torch.cat([emb,score],dim=1))

lstm_auc=train(LSTMAlone(), False)
hybrid_auc=train(Hybrid(), True)

## 3. Result

In [3]:
print(f"XGBoost only (static)     : {float(d['xgb_auc']):.4f}")
print(f"LSTM only    (sequence)   : {lstm_auc:.4f}")
print(f"HYBRID       (static+seq) : {hybrid_auc:.4f}")
print(f"hybrid vs best single     : {hybrid_auc-max(float(d['xgb_auc']),lstm_auc):+.4f}")

XGBoost only (static)     : 0.7298
LSTM only    (sequence)   : 0.7365
HYBRID       (static+seq) : 0.7750
hybrid vs best single     : +0.0385


## 4. Recap — the headline model
- **Fusion beats either branch alone** because they carry complementary information: XGBoost sees the
  static snapshot (limit, age, amounts), the LSTM sees the payment *trajectory*. Neither alone has both.
- Result: hybrid ~0.775 vs XGBoost-static 0.730 vs LSTM 0.737 — a clear lift, and the best model in
  the project so far.
- **Leakage care:** train-row XGBoost scores are out-of-fold; the fusion never trains on a score that
  saw its own label.
- The lesson of the whole project: **static ML and deep learning are complementary, not competitors.**
  Use each where it is strong (trees for tables, LSTMs for sequences) and fuse them.